In [3]:
# =============================================================================
# NOTEBOOK: 06_paper_figures.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — Publication-quality, data-driven figures (case-result figures only)
#
# This notebook renders the DATA-DRIVEN figures for the paper from artifacts
# already produced by notebooks 00–04. The four CONCEPTUAL figures (pipeline
# architecture, DSRM mapping, five-axis framework, governance layer) are
# authored separately from their JSON design specs; they are not generated here.
#
# Output figures (300 dpi, opaque background, 16:9 where applicable):
#   artifacts/figures/fig5a_sensitivity_heatmap.png
#   artifacts/figures/fig5b_reduction_waterfall.png
#   artifacts/figures/fig5c_source_provenance.png
#   artifacts/figures/fig5d_reliability_cv.png
#
# Style: academic, flat, opaque white background, no captions (added in LaTeX).
# All code and labels are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Paths + load artifacts + shared publication style
# =============================================================================
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
INFER = ARTIFACTS / "inference"
FIG = ARTIFACTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)


def rel(p):
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


def load(name):
    return json.loads((INFER / name).read_text(encoding="utf-8"))


A2 = load("agent2_time.json")
A3 = load("agent3_roi.json")
G  = load("process_graph.json")

# --- Publication style: opaque white, serif titles, restrained palette -------
rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 1.0,
    "axes.grid": False,
    "figure.dpi": 150,
})
PALETTE = {"green": "#2F6E4B", "blue": "#2F4B6E", "amber": "#B8860B",
           "red": "#B03A3A", "muted": "#5A5A5A"}
print("[INFO] Artifacts loaded; publication style set.")


# %%
# =============================================================================
# Cell 2. Figure 5a — ROI sensitivity heatmap (cases/month × hourly wage)
#
# Higher-quality version of the earlier heatmap: 16:9, break-even contour
# emphasized (ROI = 0), clean annotations.
# =============================================================================
sg = pd.DataFrame(A3["sensitivity_grid"])
base_pr = A2["params"]["partial_retain"]
sl = sg[sg["partial_retain"] == base_pr]
pivot = sl.pivot_table(index="hourly_wage_usd", columns="cases_per_month",
                       values="roi_pct")

fig, ax = plt.subplots(figsize=(12.8, 7.2))  # 16:9
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", origin="lower",
               vmin=-max(abs(pivot.values.min()), 1), vmax=pivot.values.max())
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_xlabel("cases per month", fontsize=13)
ax.set_ylabel("hourly wage (USD)", fontsize=13)
ax.set_title(f"ROI (%) sensitivity  ·  partial-retain fixed at {base_pr}")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v:,.0f}", ha="center", va="center", fontsize=10,
                color="black", fontweight="bold" if abs(v) < 15 else "normal")
# Emphasize the break-even band (|ROI| small) with an outline.
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("ROI (%)", fontsize=12)
plt.tight_layout()
p = FIG / "fig5a_sensitivity_heatmap.png"
fig.savefig(p, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] {rel(p)}")


# %%
# =============================================================================
# Cell 3. Figure 5b — Human-effort reduction waterfall (AS-IS -> TO-BE)
#
# Decompose the monthly human-effort reduction by automatability grade, making
# the ROI mechanism legible: 'full' removes effort entirely, 'partial' retains
# a fraction, 'manual' is unchanged.
# =============================================================================
est = A2["estimates"]
AI_LANES = {"system", "gpt", "ai", "ai agent"}
def is_ai(l): l = (l or "").lower(); return l in AI_LANES or "gpt" in l

asis = 0.0
removed_full = 0.0
removed_partial = 0.0
kept = 0.0
PR = A2["params"]["partial_retain"]
for e in est:
    if is_ai(e["lane"]):
        continue
    m = float(e.get("monthly_minutes", 0) or 0)
    asis += m
    g = e["grade"]
    if g == "full":
        removed_full += m
    elif g == "partial":
        removed_partial += m * (1 - PR)
        kept += m * PR
    else:
        kept += m
tobe = kept

fig, ax = plt.subplots(figsize=(12.8, 7.2))
steps = ["AS-IS\nhuman effort", "− full\n(→AI)", "− partial\n(70% saved)", "TO-BE\nhuman effort"]
vals = [asis, -removed_full, -removed_partial, tobe]
cum = [asis, asis - removed_full, asis - removed_full - removed_partial, tobe]
colors = [PALETTE["blue"], PALETTE["green"], PALETTE["amber"], PALETTE["blue"]]
# Waterfall bars.
running = 0
for i, (s, v, c) in enumerate(zip(steps, vals, colors)):
    if i == 0 or i == len(steps) - 1:
        ax.bar(i, v, color=c, edgecolor="#333")
        ax.text(i, v + asis*0.01, f"{v:,.0f}", ha="center", fontsize=11, fontweight="bold")
        running = v
    else:
        ax.bar(i, v, bottom=running, color=c, edgecolor="#333")
        ax.text(i, running + v/2, f"{v:,.0f}", ha="center", va="center",
                fontsize=11, color="white", fontweight="bold")
        running += v
ax.set_xticks(range(len(steps)), steps, fontsize=12)
ax.set_ylabel("human effort (minutes / month)", fontsize=13)
ax.set_title(f"Human-effort reduction by automatability grade  "
             f"({(1-tobe/asis)*100:.1f}% overall)")
plt.tight_layout()
p = FIG / "fig5b_reduction_waterfall.png"
fig.savefig(p, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] {rel(p)}  (AS-IS {asis:,.0f} -> TO-BE {tobe:,.0f} min/mo)")


# %%
# =============================================================================
# Cell 4. Figure 5c — Estimate provenance (Transparency axis, stacked bar)
#
# Show how many time estimates are interview-grounded (implied) vs. bare prior,
# and how many grades carry a rationale — the transparency evidence.
# =============================================================================
src = {}
for e in est:
    src[e["source"]] = src.get(e["source"], 0) + 1
labels = ["stated", "implied", "prior"]
counts = [src.get(k, 0) for k in labels]
colors = [PALETTE["green"], PALETTE["blue"], PALETTE["muted"]]

fig, ax = plt.subplots(figsize=(12.8, 7.2))
left = 0
for lab, c, col in zip(labels, counts, colors):
    ax.barh(0, c, left=left, color=col, edgecolor="#333", label=f"{lab} ({c})")
    if c:
        ax.text(left + c/2, 0, f"{lab}\n{c}", ha="center", va="center",
                color="white", fontsize=12, fontweight="bold")
    left += c
ax.set_yticks([])
ax.set_xlabel("number of time estimates", fontsize=13)
ax.set_xlim(0, sum(counts))
ax.set_ylim(-0.6, 0.6)   # tighten the bar band so the legend has clear space below
ax.set_title("Estimate provenance  ·  interview-grounded vs. prior "
             "(Transparency axis)")
# Place the legend BELOW the axes, centered, so it never overlaps the bar.
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18),
          ncol=3, frameon=False, fontsize=12)
plt.tight_layout()
p = FIG / "fig5c_source_provenance.png"
fig.savefig(p, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] {rel(p)}  source dist={src}")


# %%
# =============================================================================
# Cell 5. Figure 5d — Reliability: per-node Self-Consistency CV distribution
# =============================================================================
cvs = np.array([float(e.get("cv_minutes", 0) or 0) for e in est])
fig, ax = plt.subplots(figsize=(12.8, 7.2))
ax.hist(cvs, bins=12, color=PALETTE["blue"], edgecolor="#333", alpha=0.85)
ax.axvline(cvs.mean(), color=PALETTE["red"], linestyle="--", linewidth=2,
           label=f"mean CV = {cvs.mean():.3f}")
ax.axvline(0.25, color=PALETTE["green"], linestyle=":", linewidth=2,
           label="CV = 0.25 (stability reference)")
ax.set_xlabel("coefficient of variation across Self-Consistency rollouts", fontsize=13)
ax.set_ylabel("number of nodes", fontsize=13)
ax.set_title("Estimate reliability  ·  dispersion across rollouts "
             f"(N={A2['n_rollouts']})")
ax.legend(frameon=False)
plt.tight_layout()
p = FIG / "fig5d_reliability_cv.png"
fig.savefig(p, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[INFO] {rel(p)}  mean CV={cvs.mean():.3f}")

print("\n[INFO] Data-driven paper figures complete. Conceptual figures "
      "(pipeline, DSRM, five-axis, governance) are authored from their JSON "
      "specs separately.")

[INFO] Artifacts loaded; publication style set.
[INFO] artifacts\figures\fig5a_sensitivity_heatmap.png
[INFO] artifacts\figures\fig5b_reduction_waterfall.png  (AS-IS 41,050 -> TO-BE 19,755 min/mo)
[INFO] artifacts\figures\fig5c_source_provenance.png  source dist={'prior': 17, 'implied': 29}
[INFO] artifacts\figures\fig5d_reliability_cv.png  mean CV=0.261

[INFO] Data-driven paper figures complete. Conceptual figures (pipeline, DSRM, five-axis, governance) are authored from their JSON specs separately.
